In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
sns.set_style("whitegrid")

# Task 1: Build and Summarize Synthetic Datasets


In [ ]:
from scipy.stats import skew, kurtosis

# Dataset A — Symmetric
Dataset_A=np.random.normal(loc=50,scale=10,size=500)

#Dataset B — Right-skewed
Dataset_B=np.random.exponential(scale=20,size=500)
Dataset_B=(Dataset_B - Dataset_B.min())/(Dataset_B.max() - Dataset_B.min())*100

#Dataset C — Bimodal
part1 =np.random.normal(loc=30, scale=8,size=250)
part2 = np.random.normal(loc=70, scale=8, size=250)
Dataset_C = np.concatenate([part1, part2])

def summarize(data):
    data_series = pd.Series(data)
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    
    return {
        "Mean": round(np.mean(data), 2),
        "Median": round(np.median(data), 2),
        "Mode": round(data_series.mode().iloc[0], 2),
        "Standard Deviation": round(np.std(data, ddof=1), 2),
        "Variance": round(np.var(data, ddof=1), 2),
        "IQR": round(q3 - q1, 2),
        "Skewness": round(skew(data), 2),
        "Kurtosis": round(kurtosis(data), 2)
    }

summary_table = pd.DataFrame({
    "Dataset A (Symmetric)": summarize(Dataset_A),
    "Dataset B (Right-skewed)": summarize(Dataset_B),
    "Dataset C (Bimodal)": summarize(Dataset_C)
})

summary_table

Some statistics look similar only in some datasets, especially between Dataset A and Dataset C. For example, their mean values are very close (49.38 and 49.60), and the medians are also almost the same. If we only looked at these central values, we might think the two datasets are similar. But other statistics show that they are actually different. Dataset C has a much larger standard deviation, variance, and IQR, which means the values are more spread out. Dataset B is the most clearly different because its mean and median are much lower, and the skewness is high (1.93), showing a strong right-skewed shape. Kurtosis also helps reveal the difference: Dataset B has high kurtosis, which means heavier tails and more extreme values, while Dataset C has negative kurtosis, showing a flatter distribution. This shows that mean and median alone are not enough, and spread plus shape statistics explain the real differences better

# Task 2: Visualize the Distributions


In [ ]:
# datasets and names
datasets = [Dataset_A, Dataset_B, Dataset_C]
names = ["Dataset A (Symmetric)", "Dataset B (Right-skewed)", "Dataset C (Bimodal)"]

# create figure
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for i, data in enumerate(datasets):
    
    # Histogram + KDE
    sns.histplot(data, bins=30, kde=True, ax=axes[i, 0])
    axes[i, 0].set_title(f"{names[i]} - Histogram + KDE")
    axes[i, 0].set_xlabel("Value")
    axes[i, 0].set_ylabel("Frequency")
    
    # Boxplot horizontal
    sns.boxplot(x=data, ax=axes[i, 1])
    axes[i, 1].set_title(f"{names[i]} - Boxplot")
    axes[i, 1].set_xlabel("Value")
    
    # KDE only
    sns.kdeplot(data, ax=axes[i, 2])
    
    mean_val = np.mean(data)
    median_val = np.median(data)
    
    axes[i, 2].axvline(mean_val, color='red', linestyle='--', label='Mean')
    axes[i, 2].axvline(median_val, color='blue', linestyle='-', label='Median')
    
    axes[i, 2].set_title(f"{names[i]} - KDE")
    axes[i, 2].set_xlabel("Value")
    axes[i, 2].set_ylabel("Density")
    axes[i, 2].legend()

plt.tight_layout()
plt.show()

The mean and median diverge the most in Dataset B (Right-skewed). This is clearly visible both in the KDE plot and in the boxplot, where the red dashed mean line is positioned noticeably to the right of the blue median line. The main visual reason is the long right tail of the distribution. Most values are concentrated at lower levels, but a smaller number of large values extend far to the right and pull the mean upward. In Dataset A, the mean and median stay almost together because the distribution is symmetric. In Dataset C, they are also very close because even though the distribution is bimodal, the two peaks are balanced around the center, so the central values remain similar

# Task 3: The Outlier Stress Test


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis

# 1. Create modified copy
dataset_A_modified = Dataset_A.copy()

# find indices of 5 largest values
largest_indices = np.argsort(dataset_A_modified)[-5:]

# multiply the 5 largest values by 10
dataset_A_modified[largest_indices] = dataset_A_modified[largest_indices] * 10

# 2. Summary statistics function
def summarize(data):
    data_series = pd.Series(data)
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)

    return {
        "Mean": round(np.mean(data), 2),
        "Median": round(np.median(data), 2),
        "Mode": round(data_series.mode().iloc[0], 2),
        "Standard Deviation": round(np.std(data, ddof=1), 2),
        "Variance": round(np.var(data, ddof=1), 2),
        "IQR": round(q3 - q1, 2),
        "Skewness": round(skew(data), 2),
        "Kurtosis": round(kurtosis(data), 2)
    }

# 3. Summary table
summary_outlier_test = pd.DataFrame({
    "Original Dataset A": summarize(Dataset_A),
    "Modified Dataset A": summarize(dataset_A_modified)
})

print(summary_outlier_test)

# 4. Side-by-side boxplots
fig, ax = plt.subplots(figsize=(10, 6))

box = ax.boxplot(
    [Dataset_A, dataset_A_modified],
    vert=False,
    labels=["Original Dataset A", "Modified Dataset A"],
    patch_artist=True
)

# mean values
mean_original = np.mean(Dataset_A)
mean_modified = np.mean(dataset_A_modified)

# overlay mean as diamond marker
ax.scatter(mean_original, 1, marker='D', s=80, label='Mean')
ax.scatter(mean_modified, 2, marker='D', s=80)

ax.set_title("Outlier Stress Test: Original vs Modified Dataset A")
ax.set_xlabel("Value")
ax.set_ylabel("Dataset")
ax.legend()

plt.tight_layout()
plt.show()

The statistics that changed the most were the standard deviation, variance, skewness, and kurtosis. Standard deviation increased from 9.89 to 71.27, and variance became much larger because the extreme outliers created a very wide spread in the data. Skewness and kurtosis also changed strongly, showing that the distribution became highly right-skewed and gained very heavy tails. In contrast, the median and IQR remained almost unchanged, which shows that they are more robust to extreme values.

If I were reporting a typical value for the modified dataset, I would choose the median instead of the mean. The reason is that the mean was pulled upward by the extreme outliers and no longer represents where most values are concentrated. The median stayed near the center of the main data and still reflects the typical observation better